# Ficheros · Utilidades de texto

Programas que procesan ficheros de texto línea a línea, al estilo de las órdenes de Unix (`head`, `tail`, `wc`), ficheros de usuarios y grupos, y cifrado.

## 1. Ficheros de ejemplo

Crear los ficheros con los que trabajan los ejercicios: un texto, un `passwd` y un `group` con el formato de Unix (en Windows no existen `/etc/passwd` ni `/etc/group`).

In [1]:
texto = """En un lugar de la Mancha, de cuyo nombre no quiero acordarme,
no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero,
adarga antigua, rocín flaco y galgo corredor.
Una olla de algo más vaca que carnero, salpicón las más noches,
duelos y quebrantos los sábados, lantejas los viernes,
algún palomino de añadidura los domingos, consumían las tres partes de su hacienda.
El resto della concluían sayo de velarte, calzas de velludo para las fiestas,
con sus pantuflos de lo mesmo, y los días de entre semana
se honraba con su vellorí de lo más fino.
Tenía en su casa una ama que pasaba de los cuarenta,
y una sobrina que no llegaba a los veinte,
y un mozo de campo y plaza, que así ensillaba el rocín como tomaba la podadera.
"""
passwd = """root:x:0:0:Administrador:/root:/bin/bash
al55555:x:1000:2000:Pedro Pérez:/home/al55555:/bin/bash
al55556:x:1001:2000:Ana Gil:/home/al55556:/bin/bash
al55557:x:1002:2001:Luis Mar:/home/al55557:/bin/zsh
"""
group = "root:x:0:\ngestion:x:2000:\ninformatica:x:2001:\n"
for nombre, contenido in (("quijote.txt", texto), ("passwd", passwd), ("group", group)):
    with open(nombre, "w", encoding="utf-8") as f:
        f.write(contenido)

## 2. Contar caracteres y buscar una palabra

Contar los caracteres de un fichero (incluidos los saltos de línea) y decir si una palabra aparece en él.

In [2]:
import re

with open(input("Fichero: "), encoding="utf-8") as f:
    print(len(f.read()))
nombre, palabra = input("Fichero: "), input("Palabra: ")
with open(nombre, encoding="utf-8") as f:
    print(any(re.search(rf"{re.escape(palabra)}", linea) for linea in f))

Fichero: quijote.txt
735
Fichero: quijote.txt
Palabra: rocín
False


## 3. Líneas numeradas y líneas con una palabra

Mostrar cada línea precedida de su número; y una función que devuelva las líneas que contienen una palabra, usada en un bucle que pregunta palabras hasta que el usuario no quiera más.

In [3]:
with open(input("Fichero: "), encoding="utf-8") as f:
    for numero, linea in enumerate(f, 1):
        print(f"{numero:3} {linea}", end="")

def lineas_con(ruta, palabra):
    with open(ruta, encoding="utf-8") as f:
        return [l.rstrip("\n") for l in f if palabra in l]

ruta = input("Fichero: ")
while True:
    palabra = input("Palabra: ")
    print(*lineas_con(ruta, palabra), sep="\n")
    if input("¿Otra palabra? (s/n): ") != "s":
        break

Fichero: quijote.txt
  1 En un lugar de la Mancha, de cuyo nombre no quiero acordarme,
  2 no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero,
  3 adarga antigua, rocín flaco y galgo corredor.
  4 Una olla de algo más vaca que carnero, salpicón las más noches,
  5 duelos y quebrantos los sábados, lantejas los viernes,
  6 algún palomino de añadidura los domingos, consumían las tres partes de su hacienda.
  7 El resto della concluían sayo de velarte, calzas de velludo para las fiestas,
  8 con sus pantuflos de lo mesmo, y los días de entre semana
  9 se honraba con su vellorí de lo más fino.
 10 Tenía en su casa una ama que pasaba de los cuarenta,
 11 y una sobrina que no llegaba a los veinte,
 12 y un mozo de campo y plaza, que así ensillaba el rocín como tomaba la podadera.
Fichero: quijote.txt
Palabra: rocín
adarga antigua, rocín flaco y galgo corredor.
y un mozo de campo y plaza, que así ensillaba el rocín como tomaba la podadera.
¿Otra palabra? (s/n): s
Palabra: l

## 4. Líneas más largas

Mostrar la primera línea más larga y, después, todas las líneas más largas leyendo el fichero una sola vez.

In [4]:
with open("quijote.txt", encoding="utf-8") as f:
    mas_larga, todas = "", []
    for linea in f:
        linea = linea.rstrip("\n")
        if len(linea) > len(mas_larga):
            mas_larga, todas = linea, [linea]
        elif len(linea) == len(mas_larga):
            todas.append(linea)
print(mas_larga)
print(todas)

algún palomino de añadidura los domingos, consumían las tres partes de su hacienda.
['algún palomino de añadidura los domingos, consumían las tres partes de su hacienda.']


## 5. `head` y `tail`

Mostrar las 10 (o $n$) primeras líneas y las 10 (o $n$) últimas, leyendo el fichero una sola vez y guardando solo las últimas $n$.

In [5]:
from collections import deque
from itertools import islice

def head(ruta, n=10):
    with open(ruta, encoding="utf-8") as f:
        return [l.rstrip("\n") for l in islice(f, n)]

def tail(ruta, n=10):
    with open(ruta, encoding="utf-8") as f:
        # deque con maxlen descarta las líneas antiguas
        return [l.rstrip("\n") for l in deque(f, maxlen=n)]

print(len(head("quijote.txt")), len(tail("quijote.txt")))
print(*head("quijote.txt", int(input("n: "))), sep="\n")
print(*tail("quijote.txt", int(input("n: "))), sep="\n")

10 10
n: 3
En un lugar de la Mancha, de cuyo nombre no quiero acordarme,
no ha mucho tiempo que vivía un hidalgo de los de lanza en astillero,
adarga antigua, rocín flaco y galgo corredor.
n: 2
y una sobrina que no llegaba a los veinte,
y un mozo de campo y plaza, que así ensillaba el rocín como tomaba la podadera.


## 6. Usuarios y grupos

Con el formato de `passwd` (`login:x:uid:gid:nombre:directorio:intérprete`) y `group` (`nombre:x:gid:`): mostrar los nombres reales; dado un login, su nombre real dejando de leer en cuanto se encuentra; dado un grupo, el nombre de sus usuarios; y guardar ese listado en `<grupo>.grp`.

In [6]:
with open("passwd", encoding="utf-8") as f:
    print([linea.split(":")[4] for linea in f])

login = input("Login: ")
with open("passwd", encoding="utf-8") as f:
    for linea in f:
        campos = linea.split(":")
        if campos[0] == login:
            print(campos[4])
            break

grupo = input("Grupo: ")
with open("group", encoding="utf-8") as f:
    gid = next((l.split(":")[2] for l in f if l.split(":")[0] == grupo), None)
with open("passwd", encoding="utf-8") as f:
    nombres = [l.split(":")[4] for l in f if l.split(":")[3] == gid]
with open(f"{grupo}.grp", "w", encoding="utf-8") as f:
    f.write("\n".join(nombres) + "\n")
print(nombres, open(f"{grupo}.grp", encoding="utf-8").read().splitlines())

['Administrador', 'Pedro Pérez', 'Ana Gil', 'Luis Mar']
Login: al55556
Ana Gil
Grupo: gestion
['Pedro Pérez', 'Ana Gil'] ['Pedro Pérez', 'Ana Gil']


## 7. `wc`

Mostrar el número de bytes, palabras y líneas de un fichero.

In [7]:
import os

nombre = input("Fichero: ")
with open(nombre, encoding="utf-8") as f:
    contenido = f.read()
print(os.path.getsize(nombre), len(contenido.split()), contenido.count("\n"), nombre)

Fichero: quijote.txt
763 136 12 quijote.txt


## 8. Quitar tildes

Escribir una versión del fichero en la que cada vocal acentuada se sustituye por la misma vocal sin acentuar.

In [8]:
tabla = str.maketrans("áéíóúÁÉÍÓÚ", "aeiouAEIOU")
with open("quijote.txt", encoding="utf-8") as f, open("sin_tildes.txt", "w", encoding="utf-8") as g:
    for linea in f:
        g.write(linea.translate(tabla))
print(head("sin_tildes.txt", 2))

['En un lugar de la Mancha, de cuyo nombre no quiero acordarme,', 'no ha mucho tiempo que vivia un hidalgo de los de lanza en astillero,']


## 9. Cifrar y descifrar ficheros

`cifra.py` desplaza cada letra $n$ posiciones en el alfabeto (cifrado César). Escribir `descifra`, que recupera el original, y una función que diga si un fichero es la versión cifrada de otro (probando todos los desplazamientos).

In [9]:
from string import ascii_lowercase, ascii_uppercase

def cesar(texto, n):
    tabla = str.maketrans(ascii_lowercase + ascii_uppercase,
                          ascii_lowercase[n:] + ascii_lowercase[:n] + ascii_uppercase[n:] + ascii_uppercase[:n])
    return texto.translate(tabla)

def cifra(origen, destino, n):
    with open(origen, encoding="utf-8") as f, open(destino, "w", encoding="utf-8") as g:
        g.write(cesar(f.read(), n % 26))

def descifra(origen, destino, n):
    cifra(origen, destino, -n)

def es_version_cifrada(cifrado, original):
    with open(cifrado, encoding="utf-8") as f, open(original, encoding="utf-8") as g:
        a, b = f.read(), g.read()
    return any(cesar(b, n) == a for n in range(1, 26))

cifra("quijote.txt", "cifrado.txt", 3)
descifra("cifrado.txt", "descifrado.txt", 3)
print(head("cifrado.txt", 1), open("descifrado.txt", encoding="utf-8").read() == texto, es_version_cifrada("cifrado.txt", "quijote.txt"))

['Hq xq oxjdu gh od Pdqfkd, gh fxbr qrpeuh qr txlhur dfrugduph,'] True True


## 10. Primeros 100 primos en un fichero

Guardar los 100 primeros números primos en `primos.txt`, uno por línea.

In [10]:
primos, n = [], 2
while len(primos) < 100:
    if all(n % p for p in primos if p * p <= n):
        primos.append(n)
    n += 1
with open("primos.txt", "w", encoding="utf-8") as f:
    f.writelines(f"{p}\n" for p in primos)
print(tail("primos.txt", 3))

['521', '523', '541']
